# LSTM Sentiment Analysis
**Dataset:** `merged_sentiment.csv` (50,000 IMDB reviews + 1,600,000 tweets)

**Training sample:** 500,000 rows — all 50,000 IMDB reviews + 450,000 tweets (balanced: 250k positive / 250k negative)

**Model:** Bidirectional LSTM

---
### Instructions
1. Go to **Runtime → Change runtime type → T4 GPU** before running.
2. Upload `merged_sentiment.csv` to your Google Drive.
3. Run all cells in order.
4. Download `sentiment_lstm.keras` and `sentiment_lstm_tokenizer.json` at the end.

In [ ]:
# Check GPU is available
import tensorflow as tf
print('TensorFlow version:', tf.__version__)
print('GPU available:', tf.config.list_physical_devices('GPU'))

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# ── UPDATE THIS PATH to where you uploaded merged_sentiment.csv ──
CSV_PATH = '/content/drive/MyDrive/merged_sentiment.csv'

In [ ]:
import random
import numpy as np
import pandas as pd
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
import json, os

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
# ── Hyperparameters ──────────────────────────────────────────────
SAMPLE_SIZE  = 500_000   # total training rows (250k pos + 250k neg)
IMDB_TOTAL   = 50_000    # all IMDB rows are always included
VOCAB_SIZE   = 50_000    # top N words kept
MAX_LEN      = 100       # max tokens per review (tweets are short)
EMBED_DIM    = 128
LSTM_UNITS   = 64
BATCH_SIZE   = 256
EPOCHS       = 10
# ─────────────────────────────────────────────────────────────────

In [ ]:
# Load merged_sentiment.csv
# Columns: text (str), label (0=negative, 1=positive)
print('Loading data...')
df = pd.read_csv(CSV_PATH, dtype={'label': int})
print(f'Total rows: {len(df):,}')
print(df['label'].value_counts())

In [ ]:
# Sample: all IMDB rows + top-up with tweets
# merged_sentiment.csv layout: IMDB rows first (50,000), then tweets
imdb_df  = df.iloc[:IMDB_TOTAL]
tweet_df = df.iloc[IMDB_TOTAL:]

half         = SAMPLE_SIZE // 2
tweet_slots  = half - (imdb_df['label'] == 1).sum()
tweet_nslots = half - (imdb_df['label'] == 0).sum()

tweet_pos = tweet_df[tweet_df['label'] == 1].sample(n=int(tweet_slots),  random_state=SEED)
tweet_neg = tweet_df[tweet_df['label'] == 0].sample(n=int(tweet_nslots), random_state=SEED)

sample_df = pd.concat([imdb_df, tweet_pos, tweet_neg]).sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f'Sample size : {len(sample_df):,}')
print(sample_df['label'].value_counts())

In [ ]:
texts  = sample_df['text'].astype(str).tolist()
labels = sample_df['label'].tolist()

X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.2, random_state=SEED, stratify=labels
)
print(f'Train: {len(X_train):,}   Test: {len(X_test):,}')

In [ ]:
# Tokenise
print('Fitting tokenizer...')
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)
print(f'Vocabulary size: {len(tokenizer.word_index):,}')

X_train_seq = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=MAX_LEN, truncating='post', padding='post')
X_test_seq  = pad_sequences(tokenizer.texts_to_sequences(X_test),  maxlen=MAX_LEN, truncating='post', padding='post')

y_train_arr = np.array(y_train)
y_test_arr  = np.array(y_test)

print(f'X_train shape: {X_train_seq.shape}')
print(f'X_test  shape: {X_test_seq.shape}')

In [ ]:
# Build Bidirectional LSTM model
model = Sequential([
    Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM, input_length=MAX_LEN),
    Bidirectional(LSTM(LSTM_UNITS, return_sequences=True)),
    Dropout(0.3),
    Bidirectional(LSTM(LSTM_UNITS // 2)),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=2, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=1, verbose=1)
]

history = model.fit(
    X_train_seq, y_train_arr,
    validation_split=0.1,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks
)

In [ ]:
# Evaluate on held-out test set
loss, accuracy = model.evaluate(X_test_seq, y_test_arr, batch_size=BATCH_SIZE, verbose=0)
print(f'Test Loss     : {loss:.4f}')
print(f'Test Accuracy : {accuracy:.1%}')

# Detailed metrics
from sklearn.metrics import classification_report
y_pred = (model.predict(X_test_seq, batch_size=BATCH_SIZE) > 0.5).astype(int).flatten()
print()
print(classification_report(y_test_arr, y_pred, target_names=['Negative', 'Positive']))

In [ ]:
# Plot training history
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history.history['accuracy'],     label='Train')
ax1.plot(history.history['val_accuracy'], label='Validation')
ax1.set_title('Accuracy')
ax1.set_xlabel('Epoch')
ax1.legend()

ax2.plot(history.history['loss'],     label='Train')
ax2.plot(history.history['val_loss'], label='Validation')
ax2.set_title('Loss')
ax2.set_xlabel('Epoch')
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Save model
model.save('sentiment_lstm.keras')
print('Model saved: sentiment_lstm.keras')

# Save tokenizer
tokenizer_json = tokenizer.to_json()
with open('sentiment_lstm_tokenizer.json', 'w', encoding='utf-8') as f:
    f.write(tokenizer_json)
print('Tokenizer saved: sentiment_lstm_tokenizer.json')

In [ ]:
# Download both files to your local machine
from google.colab import files
files.download('sentiment_lstm.keras')
files.download('sentiment_lstm_tokenizer.json')

## Next steps
Once downloaded, place both files in the `data/` folder of the thesis project:
```
data/sentiment_lstm.keras
data/sentiment_lstm_tokenizer.json
```
The FastAPI backend can then load and use the LSTM model alongside the other classifiers.